In [1]:
import json
import os
import pickle

import numpy as np
# import pandas as pd
import torch
from B0_MODEL import UNET
from B1_UTILS import (identify_chain, load_checkpoint, map_normalizing,
                      map_writing, predict_location, smooth_location)
from B2_UTILS_modeling import refine_mass_centers, write_pdb  # track_5_3,

In [2]:
# ====================== Hyperparameters etc. ========================
MAIN_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Training/TEST'
MODEL_TYPES = ['secondary_strctures', 'key_atoms', 'residue_types']
MAP_DIRS_TYPES = [['map_sample'],
                  ['map_sample', 'secondary_strctures_pred_prob'],
                  [
                      'map_sample', 'secondary_strctures_pred_prob',
                      'key_atoms_pred_prob'
                  ]]
# IN_CHANNELS = [1, 6, 10]

data_path = os.path.join(MAIN_PATH, 'train_val_data')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
# batch_size (vs GPU Memory: 1vs2877; 2vs3485; 4vs4668; 8vs7162)
batch_size = 8

num_workers = 2
learning_rate = 1e-4
pin_memory = True

In [3]:
def initial_setting(model_index, data_path, device):
    """
    This function performs the initial setup before training process:
    1. Read the class weights of the UNET model,
    2. Read the saved epoch number,
    3. Generate paths of file that will be loaded to UNET model.
    """

    model_type = MODEL_TYPES[model_index]
    # Load the class weights of all model types
    weight_path = os.path.join(data_path, 'class_weight_for_training.txt')
    try:
        # Load the weight for training
        with open(weight_path, "r") as file:
            weight_all = json.load(file)

        if model_type not in weight_all.keys():
            print(f"Weight for {model_type} is not found.")
            exit()
        else:
            c_w = weight_all[model_type]
            class_weights = torch.FloatTensor(c_w).to(device)

            # Calculate the number of output channels based on the number of tags in model_type
            out_channels = len(c_w)
            in_channels = 1
            if model_index > 0:
                for ii in range(model_index):
                    in_channels = in_channels + len(
                        weight_all[MODEL_TYPES[ii]])

    except FileNotFoundError:
        print("Weight file does not exist.")
        exit()

    # Load the load_epoch_num data or initialize it if the file doesn't exist
    load_epoch_num_path = os.path.join(data_path, "epoch_num_path.txt")

    if not os.path.exists(load_epoch_num_path):
        load_epoch_num_all = {model_type: 0}
    else:
        with open(load_epoch_num_path, "r") as file:
            load_epoch_num_all = json.load(file)
            if model_type not in load_epoch_num_all.keys():
                load_epoch_num_all[model_type] = 0
    load_epoch_num = load_epoch_num_all[model_type]

    # Save the updated load_epoch_num_all data
    with open(load_epoch_num_path, "w") as file:
        json.dump(load_epoch_num_all, file)

    load_model_path = os.path.join(
        data_path, 'checkpoint', model_type,
        f'{model_type}.epoch{load_epoch_num}.pth.tar')

    return class_weights, in_channels, out_channels, load_epoch_num, load_model_path


In [4]:
RAW_DATA_PATH = '/home/qiboxu/Database/U_NET/EMDB_PDB_for_U_Net/Filtered_Dateset/Raw'
MAP_PATH = os.path.join(RAW_DATA_PATH, 'EMD-0065_re_3.27', 'emd_0065.map')
folder_path = os.path.dirname(MAP_PATH)
map_name = os.path.basename(MAP_PATH)
os.makedirs(os.path.join(folder_path, "possibility"), exist_ok=True)
os.makedirs(os.path.join(folder_path, "unrefined"), exist_ok=True)
os.makedirs(os.path.join(folder_path, "refined"), exist_ok=True)
os.makedirs(os.path.join(folder_path, "chains"), exist_ok=True)
os.makedirs(os.path.join(folder_path, "temp_in_chain"), exist_ok=True)
map_data, zoom_factors = map_normalizing(MAP_PATH)


In [ ]:
model_index = 0
model_type = MODEL_TYPES[model_index]
class_weights, in_channels, out_channels, load_epoch_num, load_model_path = initial_setting(
    model_index, data_path, device=DEVICE)
print(f"\n==> Start {model_type} prediction: ")
model = UNET(in_channels, out_channels).to(DEVICE)
load_checkpoint(load_model_path, model, map_location=DEVICE)
part_possibility = predict_location(map_data, model, out_channels, DEVICE)
model_data = np.argmax(part_possibility, 0)

In [7]:
print(part_possibility.shape)
print(map_data.shape)

(5, 200, 200, 200)
(200, 200, 200)


In [21]:
for channel_num in range(1, out_channels):
    possibility = part_possibility[channel_num, :, :, :]
    data = np.where(model_data == channel_num, 1, 0)
    print(f"\n==> Start map writing of {model_type}: ")

    out_map_file = f"{map_name.split('.mrc')[0]}_{model_type}_{channel_num}_possibility.mrc"
    out_map = os.path.join(folder_path, "possibility", out_map_file)
    map_writing(MAP_PATH, possibility, out_map, zoom_factors)

    out_map_file = f"{map_name.split('.mrc')[0]}_{model_type}_{channel_num}.mrc"
    out_map = os.path.join(folder_path, "unrefined", out_map_file)
    map_writing(MAP_PATH, data, out_map, zoom_factors, is_int=True)

    out_map_file = f"{map_name.split('.mrc')[0]}_{model_type}_{channel_num}_refined.mrc"
    out_map = os.path.join(folder_path, "refined", out_map_file)
    smooth_loc = smooth_location(data, channel_num)
    map_writing(MAP_PATH, smooth_loc, out_map, zoom_factors, is_int=True)

In [27]:
model_index = 1
model_type = MODEL_TYPES[model_index]
class_weights, in_channels, out_channels, load_epoch_num, load_model_path = initial_setting(
    model_index, data_path, device=DEVICE)
print(f"\n==> Start {model_type} prediction: ")
model = UNET(in_channels, out_channels).to(DEVICE)
load_checkpoint(load_model_path, model, map_location=DEVICE)
map_data = np.expand_dims(map_data, axis=0)
map_data = np.concatenate((map_data, part_possibility), axis=0)
part_possibility = predict_location(map_data, model, out_channels, DEVICE)
model_data = np.argmax(part_possibility, 0)


==> Start key_atoms prediction: 


ValueError: could not broadcast input array from shape (6,200,200,200) into shape (6,200,200)


==> Start map writing of secondary_strctures: 
Writing map: emd_0065.map_secondary_strctures_1_possibility.mrc
Writing map: emd_0065.map_secondary_strctures_1.mrc
Writing map: emd_0065.map_secondary_strctures_1_refined.mrc

==> Start map writing of secondary_strctures: 
Writing map: emd_0065.map_secondary_strctures_2_possibility.mrc
Writing map: emd_0065.map_secondary_strctures_2.mrc
Writing map: emd_0065.map_secondary_strctures_2_refined.mrc

==> Start map writing of secondary_strctures: 
Writing map: emd_0065.map_secondary_strctures_3_possibility.mrc
Writing map: emd_0065.map_secondary_strctures_3.mrc
Writing map: emd_0065.map_secondary_strctures_3_refined.mrc

==> Start map writing of secondary_strctures: 
Writing map: emd_0065.map_secondary_strctures_4_possibility.mrc
Writing map: emd_0065.map_secondary_strctures_4.mrc
Writing map: emd_0065.map_secondary_strctures_4_refined.mrc
